In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
FILES = {
    "applications": "applications.csv",
    "bureau": "credit_bureau.csv",
    "servicing": "loan_servicing.csv",
    "holdout": "holdout_applications.csv",
}

def load_raw(raw_dir=RAW):
    """Read each raw file; force id to text so ids like '0123456789' keep their leading zero."""
    return {name: pd.read_csv(raw_dir / fname, dtype={"id": str}, low_memory=False)
            for name, fname in FILES.items()}

data = load_raw()
apps, bureau, serv, hold = (data[k] for k in ["applications", "bureau", "servicing", "holdout"])

In [2]:
ISSUES = []

def log_issue(file, column, issue, rows, action, reason):
    """Record one data problem: where, what, how many rows, and the decision."""
    ISSUES.append({"file": file, "column": column, "issue": issue,
                   "rows": int(rows), "action": action, "reason": reason})

In [3]:
def duplicate_report(df):
    """Two kinds of duplicates: identical copies, and same id with different values."""
    exact = df.duplicated().sum()                    # whole row repeated
    conflicting = df.drop_duplicates()["id"].duplicated().sum()  # same id, values differ
    return {"rows": len(df), "unique_ids": df["id"].nunique(),
            "exact_duplicates": int(exact), "same_id_different_values": int(conflicting)}

for name, df in data.items():
    print(name, duplicate_report(df))

applications {'rows': 30600, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 600}
bureau {'rows': 30000, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 0}
servicing {'rows': 30000, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 0}
holdout {'rows': 5000, 'unique_ids': 5000, 'exact_duplicates': 0, 'same_id_different_values': 0}


In [4]:
def show_conflicts(df, n=3):
    """For a few ids that repeat with different values, show only the columns that differ."""
    deduped = df.drop_duplicates()
    ids = deduped.loc[deduped["id"].duplicated(), "id"].unique()[:n]
    for loan_id in ids:
        rows = deduped[deduped["id"] == loan_id]
        diff_cols = rows.columns[rows.nunique(dropna=False) > 1].tolist()
        print(f"\nid {loan_id} differs in: {diff_cols}")
        display(rows[["id"] + diff_cols])

show_conflicts(apps)


id 8acbab3c06 differs in: ['emp_title', 'dti']


,id,emp_title,dti
108,8acbab3c06,Hillstone,15.53
1199,8acbab3c06,HILLSTONE,15.54



id ddefd45498 differs in: ['emp_title', 'dti']


,id,emp_title,dti
429,ddefd45498,Safety Manager,15.36
1214,ddefd45498,SAFETY MANAGER,15.37



id 01edeb5c98 differs in: ['emp_title', 'dti']


,id,emp_title,dti
1083,01edeb5c98,Legal Negotiator,14.49
1506,01edeb5c98,LEGAL NEGOTIATOR,14.50


In [7]:
def conflict_summary(df):
    """For every id that repeats with different values, count which columns differ."""
    deduped = df.drop_duplicates()
    dup = deduped[deduped["id"].duplicated(keep=False)]   # all copies of repeated ids
    counts = {}
    for loan_id, rows in dup.groupby("id"):
        for col in rows.columns[rows.nunique(dropna=False) > 1]:
            counts[col] = counts.get(col, 0) + 1
    print("Repeated ids:", dup["id"].nunique())
    return pd.Series(counts, dtype=int).sort_values(ascending=False)

conflict_summary(apps)

Repeated ids: 600


dti          600
emp_title    528
dtype: int64

In [8]:
dup = apps.drop_duplicates()
dup = dup[dup["id"].duplicated(keep=False)]
g = dup.groupby("id")

case_only = g["emp_title"].agg(lambda s: s.str.lower().nunique(dropna=False) == 1)
print("emp_title differs only by case:", case_only.all())
print("Largest dti difference:", (g["dti"].max() - g["dti"].min()).max())

emp_title differs only by case: True
Largest dti difference: 0.010000000000005116


In [9]:
before = len(apps)
apps = apps.drop_duplicates(subset="id", keep="first").reset_index(drop=True)
print(f"applications: {before} → {len(apps)} rows")

log_issue(
    file="applications", column="id",
    issue="600 ids appear twice; copies differ only in emp_title capitalization (528) and dti by ≤0.01 (600)",
    rows=600,
    action="drop duplicate (keep first)",
    reason="Same loan entered twice; keeping both would double-count it. "
           "Differences are formatting/rounding, and the label and amounts never differ",
)

applications: 30600 → 30000 rows


In [5]:
def coverage(df_a, df_b, name_a, name_b):
    """Ids present in one file but not the other."""
    a, b = set(df_a["id"]), set(df_b["id"])
    print(f"{name_a} ids missing from {name_b}: {len(a - b)}")
    print(f"{name_b} ids missing from {name_a}: {len(b - a)}")

coverage(apps, bureau, "applications", "bureau")
coverage(apps, serv, "applications", "servicing")
coverage(bureau, serv, "bureau", "servicing")

applications ids missing from bureau: 0
bureau ids missing from applications: 0
applications ids missing from servicing: 0
servicing ids missing from applications: 0
bureau ids missing from servicing: 0
servicing ids missing from bureau: 0


In [6]:
overlap = set(hold["id"]) & set(apps["id"])
print("Holdout ids also in applications:", len(overlap))

Holdout ids also in applications: 0
